# 💻 Notebook do Aluno — Aula 10: Context Engineering para agentes

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 10/14 — Módulo 3: curadoria em loop agêntico · MCP overview**  
**⏱️ 1h40min**  
**🧠 Scratchpad · Context rot · MCP**  
**🔁 Andaime 55%**  

---

## 🎯 Objetivo da aula

Entender por que o contexto em agentes é um recurso ainda mais crítico que em chats simples — e dominar as estratégias de curadoria que mantêm a qualidade do agente em loops longos. Preparar o agente para a integração final da Aula 11.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime dos exercícios.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-community duckduckgo-search tiktoken matplotlib -q

import tiktoken, matplotlib.pyplot as plt
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

In [ ]:
enc = tiktoken.encoding_for_model("gpt-4")

# 👉 LACUNA 1: implemente contar_tokens_scratchpad
def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:
        texto += ___  # concatene action.log e obs
    return len(enc.encode(___))

# 👉 LACUNA 2: implemente a tool de busca com compressão de Observation
@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais. Retorna resumo conciso."""
    bruto = DuckDuckGoSearchRun().run(query)
    if len(bruto) > 500:
        return chain_resumir.invoke({___: bruto})  # invocar chain de resumo
    return bruto

# 👉 LACUNA 3: rodar as duas versões (sem e com compressão) com 5 perguntas cada
PERGUNTAS = [___, ___, ___, ___, ___]  # 5 perguntas reais do domínio

tok_sem, tok_com = [], []
for q in PERGUNTAS:
    r_sem = executor_sem.invoke({"input":q})  # sem compressão (Aula 09)
    r_com = executor_com.invoke({"input":q})  # com compressão (esta aula)
    tok_sem.append(contar_tokens_scratchpad(r_sem["intermediate_steps"]))
    tok_com.append(contar_tokens_scratchpad(___))

# 👉 LACUNA 4: plotar comparação tokens sem vs. com compressão
x = range(len(PERGUNTAS))
plt.bar([i-.2 for i in x], ___, .4, label="Sem compressão", color="salmon")
plt.bar([i+.2 for i in x], ___, .4, label="Com compressão",  color="steelblue")
plt.legend(); plt.xlabel("Pergunta"); plt.ylabel("Tokens no scratchpad")
plt.title("Impacto da compressão de Observations no scratchpad"); plt.show()

---

## ✍️ Suas anotações

Registre aqui as observações da aula (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 10

Quatro exercícios práticos em sequência — orçamento de contexto com tiktoken, compressão de Observations, agente como cliente MCP e memória episódica — sempre com perguntas do domínio do grupo.

Grupo 3–4 · Google Colab: calcule a redução média de tokens (1 - mean(tok_com)/mean(tok_sem)) × 100 e documente em célula markdown.


### Exercício 1 — Orçamento de contexto: medidor de scratchpad com tiktoken · ★★☆ · 10 min

*Individual · Colab*

1. Complete o tokenizer do medidor de tokens.
2. Complete a concatenação de `action.log` + `obs` e a contagem final.
3. Rode as 3 perguntas e leia, iteração a iteração, tool, `obs_len` e tokens acumulados — qual tool você comprimiria primeiro?

> **💡 Dica:** `intermediate_steps` é lista de tuplas `(action, observation)` — e `action.log` já vem com o bloco `Thought: ... Action: ...` formatado.


In [ ]:
# Exercício 1 — medidor de scratchpad com tiktoken
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e1", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

executor_sem = AgentExecutor(
    agent=create_react_agent(llm, tools, prompt_react),
    tools=tools, verbose=False, max_iterations=5,
    handle_parsing_errors=True,
)

# 👉 LACUNA 1: tokenizer para contar tokens do contexto
enc = tiktoken.encoding_for_model(___)

def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:
        # 👉 LACUNA 2: Thought (action.log) + Observation, separados por 

        texto += ___
    # 👉 LACUNA 3: conte os tokens do texto
    return len(enc.___(texto))

PERGUNTAS = [
    "Qual é a cláusula de garantia no documento?",
    "Qual é a cotação do dólar hoje?",
    "Qual o prazo de garantia em dias (meses × 30)?",
]
for q in PERGUNTAS:
    r = executor_sem.invoke({"input": q})
    print(f"\nPergunta: {q}")
    for i, (action, obs) in enumerate(r["intermediate_steps"], 1):
        acumulado = contar_tokens_scratchpad(r["intermediate_steps"][:i])
        print(f"  Iteração {i}: tool={action.tool:22s} "
              f"obs_len={len(str(obs)):4d} · tokens_acumulados={acumulado}")

# Leitura típica: a iteração de busca web é a mais pesada (obs_len 800–1200
# chars ≈ 200–300 tokens); a da calculadora tem obs_len de 2–5 chars.


### Exercício 2 — Compressão de Observations: medir o ganho · ★★☆ · 10 min

*Individual · Colab*

1. Complete o parser da chain de resumo.
2. Complete a tool comprimida: o limite em chars e a chamada que resume ANTES de voltar ao scratchpad.
3. Rode as mesmas perguntas nos dois executores (sem vs. com compressão) e calcule a redução média.

> **💡 Dica:** comprimir dentro da tool (antes de voltar ao scratchpad) é mais barato que resumir o scratchpad inteiro depois — a Observation é o maior vilão do orçamento.


In [ ]:
# Exercício 2 — ganho da compressão de Observations
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e2", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

import statistics
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

enc = tiktoken.encoding_for_model("gpt-4")

def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:
        texto += f"{action.log}\n{obs}\n"
    return len(enc.encode(texto))

# 👉 LACUNA 1: parser de string no fim da chain de resumo
chain_resumir = (
    ChatPromptTemplate.from_template(
        "Resuma em no máximo 3 frases, preservando APENAS os fatos essenciais:\n\n{texto}")
    | llm | ___()
)

# 👉 LACUNA 2: limite em chars para comprimir
# 👉 LACUNA 3: invoque a chain de resumo com a chave {texto}
@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais na web. Retorna resumo conciso."""
    bruto = DuckDuckGoSearchRun().run(query)
    if len(bruto) > ___:
        return chain_resumir.___({"texto": bruto})
    return bruto

tools_sem = [buscar_nos_documentos, buscar_na_web, calcular]
tools_com = [buscar_nos_documentos, buscar_na_web_comprimida, calcular]
executor_sem = AgentExecutor(
    agent=create_react_agent(llm, tools_sem, prompt_react),
    tools=tools_sem, max_iterations=5, handle_parsing_errors=True)
executor_com = AgentExecutor(
    agent=create_react_agent(llm, tools_com, prompt_react),
    tools=tools_com, max_iterations=5, handle_parsing_errors=True)

PERGUNTAS = [
    "Qual é a cotação do dólar hoje?",
    "Quais são as últimas notícias sobre o tema do domínio?",
    "Qual o prazo de garantia em dias (meses × 30)?",
]
tok_sem, tok_com = [], []
for q in PERGUNTAS:
    tok_sem.append(contar_tokens_scratchpad(
        executor_sem.invoke({"input": q})["intermediate_steps"]))
    tok_com.append(contar_tokens_scratchpad(
        executor_com.invoke({"input": q})["intermediate_steps"]))

reducao = (1 - statistics.mean(tok_com) / statistics.mean(tok_sem)) * 100
print(f"Sem compressão : {tok_sem}")
print(f"Com compressão : {tok_com}")
print(f"Redução média  : {reducao:.1f}%")
# Típico: ~70–80% de redução nas buscas web — o ponto a conferir é se o
# resumo preservou os fatos essenciais (números, datas).


### Exercício 3 — MCP: seu agente como cliente MCP · ★★☆ · 10 min

*Individual · Colab*

1. Complete o handshake da sessão MCP.
2. Complete o discovery das tools do servidor filesystem.
3. Complete o executor só com as tools MCP e rode a listagem de /content.

> **💡 Dica:** no MCP o servidor publica name + description + schema e o cliente apenas descobre — o agente não implementa nada, só carrega as tools.


In [ ]:
# Exercício 3 — agente como cliente MCP
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb langchain-mcp-adapters

from langchain_ollama import ChatOllama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm = ChatOllama(model="gpt-oss:120b", temperature=0)
prompt_react = hub.pull("hwchase17/react")

!node --version > /dev/null 2>&1 || apt-get install -y -qq nodejs npm > /dev/null 2>&1

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession
from mcp.client.stdio import stdio_client

async def conectar_mcp():
    async with stdio_client(
        {"command": "npx",
         "args": ["@modelcontextprotocol/server-filesystem", "/content"]}
    ) as (read, write):
        async with ClientSession(read, write) as session:
            # 👉 LACUNA 1: handshake MCP
            await session.___()
            # 👉 LACUNA 2: discovery das tools do servidor
            mcp_tools = await load_mcp_tools(___)
            print(f"Tools MCP descobertas: {[t.name for t in mcp_tools]}")
            return mcp_tools

mcp_tools = await conectar_mcp()
# 👉 LACUNA 3: executor só com as tools MCP
executor_mcp = AgentExecutor(
    agent=create_react_agent(llm, mcp_tools, prompt_react),
    tools=___,
    verbose=True, max_iterations=5,
)
print(executor_mcp.invoke(
    {"input": "Liste os arquivos em /content e mostre as 5 primeiras linhas do primeiro .txt."}
)["output"])
# Arquitetura: a tool roda no SERVIDOR MCP (processo externo) — o agente é
# apenas cliente; name/description/schema vêm do servidor, não do seu código.


### Exercício 4 — Memória episódica: persistir entre sessões · ★★☆ · 10 min

*Individual · Colab*

1. Complete a coleção episódica separada e o tipo do episódio no metadata.
2. Complete o filtro por tipo na tool de lembrança.
3. Complete o roster com a 4ª tool, salve um episódio e pergunte "O que eu perguntei na sessão anterior?".

> **💡 Dica:** memória episódica é busca semântica sobre conversas passadas — o filtro por `tipo` evita misturar episódios com os documentos do domínio.


In [ ]:
# Exercício 4 — memória episódica entre sessões
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e4", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

from datetime import datetime
from langchain_core.documents import Document

# 👉 LACUNA 1: coleção episódica separada dos documentos do domínio
db_episodico = Chroma(persist_directory=___, embedding_function=embeddings)

def salvar_sessao(pergunta: str, resposta: str, session_id: str):
    db_episodico.add_documents([Document(
        page_content=f"Pergunta: {pergunta}\nResposta: {resposta}",
        # 👉 LACUNA 2: tipo do episódio no metadata
        metadata={"session_id": session_id, "tipo": ___,
                  "timestamp": datetime.now().isoformat()},
    )])

@tool
def lembrar_sessoes_anteriores(query: str) -> str:
    """Use quando o usuário se referir a algo discutido em conversa anterior.
    Retorna resumos de interações passadas relevantes para a query."""
    # 👉 LACUNA 3: filtro pelo tipo de episódio
    docs = db_episodico.similarity_search(query, k=2, filter={"tipo": ___})
    return "\n\n".join(d.page_content for d in docs) if docs else "Nada encontrado."

# 👉 LACUNA 4: roster com a 4ª tool
tools4    = [buscar_nos_documentos, buscar_na_web, calcular, ___]
agente4   = create_react_agent(llm, tools4, prompt_react)
executor4 = AgentExecutor(agent=agente4, tools=tools4, verbose=True,
                          max_iterations=5, handle_parsing_errors=True)

# Ciclo de validação: salvar episódio → nova sessão → agente aciona a memória
salvar_sessao("Qual o prazo de garantia?", "24 meses (pág. 15)", "sessao-1")
print(executor4.invoke({"input": "O que eu perguntei na sessão anterior?"})["output"])


## 📚 Referências da aula

- Blog Anthropic Engineering — "Context Engineering for AI Agents" (setembro 2025). Fonte primária desta aula — princípio da ação mínima, tipos de memória, curadoria de scratchpad. anthropic.com/engineering/context-engineering
- Docs Model Context Protocol — Especificação oficial, servidores disponíveis e guia de implementação. modelcontextprotocol.io
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. A base empírica do context rot em contextos longos. arxiv.org/abs/2307.03172
- Docs LangChain MCP Adapters — Integrar servidores MCP como tools LangChain. github.com/langchain-ai/langchain-mcp-adapters
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes e ambientes: a analogia memória=RAM / conhecimento=HD que fundamenta os 3 tipos de memória agêntica.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 8: Memory Management — a distinção Short-Term vs. Long-Term por trás do scratchpad e da memória episódica desta aula.

---

**Próxima Aula — Aula 11 · 26/10** — Aula Integradora — Agente com RAG + Gradio ao vivo
  
100% lab. Integrar tudo. Publicar URL pública. Entregar CKP03.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*